In [7]:
import os
import shutil
import subprocess

In [8]:
ROOT = (
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], stderr=subprocess.STDOUT
    )
    .decode("utf-8")
    .strip()
)

RAW_DATA_DIR = os.path.join(ROOT, "data", "raw")
QASMBENCH_REPO_DIR = os.path.join(RAW_DATA_DIR, "QASMBench")

BENCHMARK_DIR = os.path.join(ROOT, "benchmarks")
QASMBENCH_BENCHMARK_DIR = os.path.join(BENCHMARK_DIR, "qasmbench")

DOWNLOAD_DIRECTORIES = ["small", "medium", "large"]

RAW_DATA_DIR = os.path.join(ROOT, "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(BENCHMARK_DIR, exist_ok=True)

In [9]:
REPO_URL = "https://github.com/pnnl/QASMBench.git"
COMMIT_SHA = "357b942"

In [10]:
print(f"Cloning {REPO_URL}...")
subprocess.run(["git", "clone", "--quiet", REPO_URL, QASMBENCH_REPO_DIR], check=True)

print(f"Checking out version: {COMMIT_SHA}...")
subprocess.run(["git", "-C", QASMBENCH_REPO_DIR, "checkout", "--quiet", COMMIT_SHA], check=True)


Cloning https://github.com/pnnl/QASMBench.git...
Checking out version: 357b942...


CompletedProcess(args=['git', '-C', '/home/javier/Documents/Graduate/Quantum_WPI/Quantum_Computing/Quantum_Circuit_Benchmark/qceval-internal/data/raw/QASMBench', 'checkout', '--quiet', '357b942'], returncode=0)

In [11]:
if os.path.exists(QASMBENCH_BENCHMARK_DIR):
    print("Cleaning old benchmarks...")
    shutil.rmtree(QASMBENCH_BENCHMARK_DIR)
os.makedirs(QASMBENCH_BENCHMARK_DIR, exist_ok=True)

# TODO no need to remove them if they already

print("Extracting benchmarks...")
for directory in DOWNLOAD_DIRECTORIES:
    source_path = os.path.join(QASMBENCH_REPO_DIR, directory)
    if not os.path.exists(source_path):
        continue

    target_directory_path = os.path.join(QASMBENCH_BENCHMARK_DIR, directory)
    os.makedirs(target_directory_path, exist_ok=True)

    for dirpath, _, _ in os.walk(source_path):
        circuit_name = os.path.basename(dirpath)
        qasm_file = os.path.join(dirpath, f"{circuit_name}.qasm")

        if os.path.isfile(qasm_file):
            relative_dir = os.path.relpath(dirpath, source_path)
            target_dir = os.path.join(target_directory_path, relative_dir)
            os.makedirs(target_dir, exist_ok=True)
            shutil.copy2(qasm_file, os.path.join(target_dir, f"{circuit_name}.qasm"))

    print(f"Copied .qasm files from '{directory}' directory.")

Cleaning old benchmarks...
Extracting benchmarks...
Copied .qasm files from 'small' directory.
Copied .qasm files from 'medium' directory.
Copied .qasm files from 'large' directory.


In [12]:
license_source = os.path.join(QASMBENCH_REPO_DIR, "LICENSE")
shutil.copy2(license_source, QASMBENCH_BENCHMARK_DIR)
print("Copied repository license.")

Copied repository license.


In [13]:
print("Cleaning up temporary files...")
# TODO remove?
shutil.rmtree(QASMBENCH_REPO_DIR)

Cleaning up temporary files...


In [14]:
with open(os.path.join(QASMBENCH_BENCHMARK_DIR, "VERSION_INFO.txt"), "w") as f:
    f.write(
        f"""Source: {REPO_URL}
Commit SHA: {COMMIT_SHA}
Downloaded on: {subprocess.getoutput("date")}
Directories: {", ".join(DOWNLOAD_DIRECTORIES)}
"""
    )